⏱️ **Time required:** ~10 minutes | **Type:** Interactive tutorial

# Integrations — dbt, dlt, Streaming, Databases, Cloud & More

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/06_integrations.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/06_integrations.ipynb)

Reuse your dbt schema definitions, plug into 100+ dlt sources, stream live data from WebSockets/Kafka, and extract directly from SQL databases — all with contract-driven quality gates.

In [1]:
# Install lakelogic
!pip install -q lakelogic[polars,duckdb]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

lakelogic v1.21.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb" # 'polars' , 'spark'

---
## 1. dbt Adapter — Reuse Your Schema Definitions

**The Problem:** You already have 200 models defined in dbt `schema.yml`. Rewriting them as LakeLogic contracts doubles your maintenance burden.

**The Solution:** `DataProcessor.from_dbt()` reads your dbt schema and creates a LakeLogic contract from it. Zero duplication.

In [2]:
from pathlib import Path

# Write a realistic dbt schema.yml
dbt_schema = """
version: 2
models:
  - name: customers
    description: Customer master table
    columns:
      - name: customer_id
        description: Primary key
        tests:
          - not_null
          - unique
      - name: email
        description: Customer email address
        tests:
          - not_null
      - name: first_name
        description: First name
      - name: last_name
        description: Last name
      - name: country
        description: ISO country code
        tests:
          - accepted_values:
              values: ['US', 'GB', 'DE', 'FR', 'JP']
      - name: created_at
        description: Account creation timestamp
        tests:
          - not_null
"""
Path("dbt_schema.yml").write_text(dbt_schema)

# Create a LakeLogic processor from dbt definitions
proc = ll.DataProcessor.from_dbt("dbt_schema.yml", model="customers")
print("Contract created from dbt schema:")
print(f"  Dataset: {proc.contract.dataset}")
print(f"  Fields:  {[f.name for f in proc.contract.model.fields]}")
print(f"  Rules:   {len(proc.contract.quality.row_rules)} row rules")

Contract created from dbt schema:
  Dataset: customers
  Fields:  ['customer_id', 'email', 'first_name', 'last_name', 'country', 'created_at']
  Rules:   1 row rules


In [3]:
# The Proof — generate data and run through the dbt-derived contract
gen = ll.DataGenerator.from_dbt("dbt_schema.yml", model="customers")
source_df = gen.generate(rows=500, invalid_ratio=0.08, output_format=ENGINE)

good, bad = proc.run(source_df)
good, bad = s.to_polars(good), s.to_polars(bad)
s.assert_reconciliation(source_df, good, bad)
print("\ndbt not_null + accepted_values tests → LakeLogic quality rules. Zero rewrite.")

2026-04-28 06:52:57.112 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: customers
2026-04-28 06:52:57.114 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 460 valid + 40 invalid = 500 total
2026-04-28 06:52:57.115 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:52:57.115 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:52:57.333 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 500 records built
2026-04-28 06:52:57.333 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 91 across 6 categories
2026-04-28 06:52:57.335 | INFO     | lakelogic.core.generator:generate:3506 -      EMPTY_STRING                     38 injections
2026-04-28 06:52:57.336 | INFO     | lakelogic.core.generator:generate:3506 -      NOT_NULL_VIOLATION     

source=500  good=465  bad=35
500 == 465 + 35 -> True

dbt not_null + accepted_values tests → LakeLogic quality rules. Zero rewrite.


---
## 2. dlt Adapter — Contract-Driven API Ingestion

**The Problem:** You ingest from GitHub, Stripe, Shopify and 100+ APIs via [dlt](https://dlthub.com). Data arrives with no schema enforcement — bad records flow straight into your warehouse.

**The Solution:** Declare the API directly in your contract's `source.type: dlt` block. LakeLogic extracts the data via dlt's REST API engine, then validates every row through your model and quality rules — all in one `proc.run_source()` call.

In [4]:
# Install dlt (if not already installed)
import subprocess
import sys

try:
    import dlt
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dlt"])
    import dlt
print(f"dlt v{dlt.__version__} ready")

dlt v1.24.0 ready


In [5]:
# ── The contract declares the API source directly ─────────────
# No separate dlt script needed — the contract IS the config.
github_contract = s.write_contract(
    """
version: 1.0.0
dataset: github_issues
info:
  title: bronze_github_issues
  domain: engineering
  target_layer: bronze

source:
  type: dlt
  dlt:
    base_url: https://api.github.com
    credentials: {}
    endpoints:
      - name: issues
        path: repos/dlt-hub/dlt/issues
        params:
          state: open
          per_page: 30

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: repository_url
      type: string
    - name: number
      type: integer
      required: true
    - name: title
      type: string
      required: true
    - name: body
      type: string
    - name: state
      type: string
    - name: url
      type: string
    - name: created_at
      type: string
    - name: updated_at
      type: string

quality:
  row_rules:
    - name: valid_state
      sql: "state IN ('open', 'closed')"
    - name: has_title
      sql: "title IS NOT NULL AND title != ''"

server:
  type: local
  path: "."
  schema_policy:
    evolution: "allow"
    unknown_fields: "drop"

""",
    "06_integrations_demo/github_issues.yaml",
)

print("Contract written with source.type=dlt")
print("  API: https://api.github.com/repos/dlt-hub/dlt/issues")
print("  Fields: id, repository_url, number, title, body, state, ...")
print("  Rules: valid_state, has_title")

Contract written with source.type=dlt
  API: https://api.github.com/repos/dlt-hub/dlt/issues
  Fields: id, repository_url, number, title, body, state, ...
  Rules: valid_state, has_title


In [6]:
# ── One call: dlt extraction + LakeLogic validation ────────────
# run_source() detects source.type=dlt and:
#   1. Builds a dlt REST API pipeline from the contract config
#   2. Extracts data from the GitHub API
#   3. Converts to Polars DataFrame
#   4. Runs schema validation + quality rules
#   5. Returns good/bad split with reconciliation guarantee

proc = ll.DataProcessor(github_contract, engine=ENGINE)
good, bad = proc.run_source()
good, bad = s.to_polars(good), s.to_polars(bad)
r = proc.last_report

counts = r.get("counts", {})
print("Contract-driven dlt results:")
print(f"  Source  : {counts.get('source', '?')} issues from GitHub API")
print(f"  Good    : {counts.get('good', '?')} (passed all rules)")
print(f"  Bad     : {counts.get('quarantined', '?')} (quarantined)")
print(
    f"  Match   : {counts.get('source', 0)} == {counts.get('good', 0)} + {counts.get('quarantined', 0)} -> {counts.get('source', 0) == counts.get('good', 0) + counts.get('quarantined', 0)}"
)
print("\nThe contract IS the config. No dlt script. No manual DataFrame wrangling.")

2026-04-28 06:52:58.793 | INFO     | lakelogic.core.processor:_run_dlt_source:4148 - Running dlt source for contract: bronze_github_issues
2026-04-28 06:52:58.817 | INFO     | lakelogic.adapters.dlt_adapter:_run_rest_api:219 - dlt: running REST API source from https://api.github.com
2026-04-28 06:53:05,377|[WARNING]|31516|27640|dlt|validate.py|verify_normalized_table:113|In schema `rest_api`: The following columns in table 'issues' did not receive any data during this load and therefore could not have their types inferred:
  - active_lock_reason
  - assignee
  - closed_at
  - closed_by
  - milestone
  - milestone__closed_at
  - milestone__due_on
  - performed_via_github_app
  - pinned_comment
  - pull_request__merged_at
  - type

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'active_lock_reason': {'data_type'

Contract-driven dlt results:
  Source  : 694 issues from GitHub API
  Good    : 345 (passed all rules)
  Bad     : 349 (quarantined)
  Match   : 694 == 345 + 349 -> True

The contract IS the config. No dlt script. No manual DataFrame wrangling.


In [7]:
print("preview live github data - GOOD DATA")
good.limit(3)

preview live github data - GOOD DATA


id,repository_url,number,title,body,state,url,created_at,updated_at
i64,str,i64,str,str,str,str,str,str
4337452413,"""https://api.github.com/repos/dlt-hub/dlt""",3896,"""feat: handle cross-batch schema evolution in ArrowToParquetWriter (#3…""","""### Description `arrow_concat_promote_options` currently only handles type mismatches within a single flush batch (via `pa.concat_tables`). But `pyarrow.ParquetWriter` locks its schema on the first…","""open""","""https://api.github.com/repos/dlt-hub/dlt/issues/3896""","""2026-04-27 16:56:23.000000+00:00""","""2026-04-27 21:19:48.000000+00:00"""
4337358316,"""https://api.github.com/repos/dlt-hub/dlt""",3895,"""ParquetWriter rejects cross-batch type mismatches that arrow_concat_promote_options should handle""","""### Feature description Extend `arrow_concat_promote_options` to handle type mismatches across flush batch boundaries, not just within a single batch. Currently, `ArrowToParquetWriter` uses `pa.conc…","""open""","""https://api.github.com/repos/dlt-hub/dlt/issues/3895""","""2026-04-27 16:41:17.000000+00:00""","""2026-04-27 16:42:34.000000+00:00"""
4323217928,"""https://api.github.com/repos/dlt-hub/dlt""",3892,"""Data item normalizer type validation is overly strict""","""### dlt version 1.25.0 ### Describe the problem Some operations on `DltSource` only work if the normalizer is configured as the default `dlt.common.normalizers.json.relational`. For example, access…","""open""","""https://api.github.com/repos/dlt-hub/dlt/issues/3892""","""2026-04-24 13:26:17.000000+00:00""","""2026-04-24 13:26:54.000000+00:00"""


In [8]:
print("preview live github data - BAD DATA")
bad.limit(3)

preview live github data - BAD DATA


url,repository_url,id,number,title,state,created_at,updated_at,body,_lakelogic_errors,_lakelogic_categories,quarantine_state,quarantine_reprocessed
str,str,i64,i64,str,str,str,str,str,list[str],list[str],str,bool
"""https://api.github.com/users/rudolfix""",null,17202864,null,null,null,null,null,null,"[""Rule failed: number_required (""number"" IS NOT NULL)"", ""Rule failed: title_required (""title"" IS NOT NULL)"", … ""Rule failed: has_title (title IS NOT NULL AND title != '')""]","[""completeness"", ""completeness"", … ""correctness""]","""active""",false
"""https://api.github.com/users/burnash""",null,264674,null,null,null,null,null,null,"[""Rule failed: number_required (""number"" IS NOT NULL)"", ""Rule failed: title_required (""title"" IS NOT NULL)"", … ""Rule failed: has_title (title IS NOT NULL AND title != '')""]","[""completeness"", ""completeness"", … ""correctness""]","""active""",false
"""https://api.github.com/users/elviskahoro""",null,29553206,null,null,null,null,null,null,"[""Rule failed: number_required (""number"" IS NOT NULL)"", ""Rule failed: title_required (""title"" IS NOT NULL)"", … ""Rule failed: has_title (title IS NOT NULL AND title != '')""]","[""completeness"", ""completeness"", … ""correctness""]","""active""",false


---
## 3. Live Streaming Data — Native Connectors

**The Problem:** Data arrives continuously from real-time feeds (crypto prices, wiki edits, clickstreams via Kafka). You need to validate every message against your schema before it lands in the lakehouse.

**The Solution:** LakeLogic ships with native streaming connectors (`SSEConnector`, `WebSocketConnector`, `KafkaConnector`, `WebhookConnector`, and more). Each connector’s `.stream()` method yields JSON events which you buffer into a DataFrame and pipe directly through your contract. Below we use the **Binance Public REST API** to fetch live BTC trades, then validate them — the exact same pattern works with any connector.

In [9]:
# ── Fetch live BTC trades using LakeLogic's WebSocketConnector ───
from lakelogic.engines.streaming_connectors import WebSocketConnector
import polars as pl

connector = WebSocketConnector(url="wss://stream.binance.com:9443/ws/btcusdt@trade")

# Collect 20 live trades, then close the connection
trades = []
for event in connector.stream():
    trades.append(event)
    if len(trades) >= 20:
        break
connector.close()

live_df = pl.DataFrame(trades)
print(f"Captured {live_df.height} live BTC trades via WebSocketConnector")
display(live_df.head(3))

2026-04-28 06:53:05.578 | INFO     | lakelogic.engines.streaming_connectors:stream:170 - Connecting to WebSocket: wss://stream.binance.com:9443/ws/btcusdt@trade
2026-04-28 06:53:06.623 | INFO     | lakelogic.engines.streaming_connectors:on_open:190 - ✅ Connected to WebSocket
2026-04-28 06:53:06.921 | WARNING  | lakelogic.engines.streaming_connectors:on_close:186 - WebSocket closed: None - None
2026-04-28 06:53:07.201 | INFO     | lakelogic.engines.streaming_connectors:close:227 - WebSocket connection closed


Captured 20 live BTC trades via WebSocketConnector


e,E,s,t,p,q,T,m,M
str,i64,str,i64,str,str,i64,bool,bool
"""trade""",1777355586620,"""BTCUSDT""",6257227841,"""76932.07000000""","""0.00007000""",1777355586618,false,true
"""trade""",1777355586620,"""BTCUSDT""",6257227842,"""76932.07000000""","""0.00007000""",1777355586618,false,true
"""trade""",1777355586620,"""BTCUSDT""",6257227843,"""76932.07000000""","""0.05199000""",1777355586618,false,true


In [10]:
# ── Define a contract for real-time trade validation ─────────
# Binance WebSocket sends cryptic field names (e, E, s, t, p, q, T, m, M).
# The contract uses a `pre` transformation to rename them to business-friendly names
# BEFORE validation runs.
stream_contract = s.write_contract(
    """
version: 1.0.0
dataset: btc_trades
info:
  title: bronze_btc_trades
  domain: market_data
  target_layer: bronze

source:
  type: stream
  path: wss://stream.binance.com:9443/ws/btcusdt@trade

transformations:
  - phase: pre
    rename:
      mappings:
        t: trade_id
        p: price
        q: quantity
        s: symbol
        T: trade_time
        m: is_buyer_maker
        e: event_type
        E: event_time
        M: is_best_match

model:
  fields:
    - name: trade_id
      type: integer
      required: true
    - name: price
      type: string
      required: true
    - name: quantity
      type: string
      required: true
    - name: symbol
      type: string
      required: true
    - name: trade_time
      type: integer
    - name: is_buyer_maker
      type: boolean
    - name: event_type
      type: string
    - name: event_time
      type: integer

quality:
  row_rules:
    - name: positive_price
      sql: "CAST(price AS DOUBLE) > 0"
    - name: positive_qty
      sql: "CAST(quantity AS DOUBLE) > 0"
    - name: valid_symbol
      sql: "symbol = 'BTCUSDT'"

server:
  type: local
  path: "."
  schema_policy:
    evolution: "allow"
    unknown_fields: "drop"
""",
    "06_integrations_demo/btc_trades.yaml",
)

print("Contract written with pre-validation rename transformations!")

Contract written with pre-validation rename transformations!


In [11]:
# ── Validate live trades through the contract ───────────
proc = ll.DataProcessor(stream_contract, engine=ENGINE)
good, bad = proc.run(live_df)
good, bad = s.to_polars(good), s.to_polars(bad)

s.assert_reconciliation(live_df, good, bad)
print("\nLive BTC trades validated in real-time!")
print(f"  Good : {good.height} trades passed all rules")
print(f"  Bad  : {bad.height} trades quarantined")
display(good.select(["trade_id", "symbol", "price", "quantity", "is_buyer_maker"]).head(5))

2026-04-28 06:53:07.232 | INFO     | lakelogic.core.processor:run:818 - Run complete [domain=market_data, layer=bronze] | Source: 20 | Total: 20 | Good: 20 | Quarantine: 0 | Ratio: 0.00%
2026-04-28 06:53:07.233 | WARNING  | lakelogic.core.processor:run:1026 - Schema drift detected for 'bronze_btc_trades': missing=[], unknown=['is_best_match']


source=20  good=20  bad=0
20 == 20 + 0 -> True

Live BTC trades validated in real-time!
  Good : 20 trades passed all rules
  Bad  : 0 trades quarantined


trade_id,symbol,price,quantity,is_buyer_maker
i64,str,str,str,bool
6257227841,"""BTCUSDT""","""76932.07000000""","""0.00007000""",false
6257227842,"""BTCUSDT""","""76932.07000000""","""0.00007000""",false
6257227843,"""BTCUSDT""","""76932.07000000""","""0.05199000""",false
6257227844,"""BTCUSDT""","""76932.07000000""","""0.00007000""",false
6257227845,"""BTCUSDT""","""76932.07000000""","""0.00007000""",false


---
## Setting up a Live Database for this Demo
To prove LakeLogic natively executes SQL over the wire, we will quickly create a local SQLite database and populate it with tables. (LakeLogic uses exactly the same engine logic for Postgres, MySQL, SQL Server, etc).

In [12]:
import os

abs_db_path = os.path.abspath("demo.db").replace("\\", "/")
import sqlite3
from datetime import datetime

conn = sqlite3.connect("demo.db")
c = conn.cursor()

# Seed users table (For Section 3)
c.execute("CREATE TABLE IF NOT EXISTS pg_users (id INTEGER, email TEXT, signup_date TEXT)")
c.execute("DELETE FROM pg_users")
c.executemany(
    "INSERT INTO pg_users VALUES (?, ?, ?)",
    [(1, "test@example.com", "2024-01-01"), (2, "invalid_email.com", "2024-01-02")],
)

# Seed cdc_orders table (For Section 5)
c.execute("CREATE TABLE IF NOT EXISTS cdc_orders (id INTEGER, updated_at TEXT)")
c.execute("DELETE FROM cdc_orders")
c.executemany("INSERT INTO cdc_orders VALUES (?, ?)", [(1, "2024-04-10T12:00:00Z"), (2, "2024-04-12T12:00:00Z")])

# Seed massive_orders table (For Section 6)
c.execute("CREATE TABLE IF NOT EXISTS massive_orders (id INTEGER, total REAL)")
c.execute("DELETE FROM massive_orders")
c.executemany("INSERT INTO massive_orders VALUES (?, ?)", [(i, i * 1.5) for i in range(1, 1001)])

# Seed wide_orders_table (For Section 7)
c.execute("CREATE TABLE IF NOT EXISTS wide_orders_table (order_id TEXT, total_amount REAL, huge_json TEXT)")
c.execute("DELETE FROM wide_orders_table")
c.executemany(
    "INSERT INTO wide_orders_table VALUES (?, ?, ?)",
    [("A1", 100.5, '{"data":"blob"}'), ("A2", 50.0, '{"data":"blob"}')],
)

conn.commit()
conn.close()
print("demo.db initialized with seed data!")

demo.db initialized with seed data!


---
## 4. Native Database Ingestion (SQLite / Postgres)

**The Problem:** You need to mirror a transactional database (e.g. `users` or `orders` tables) without maintaining brittle JDBC extraction layers or deploying heavy extraction pipelines.

**The Solution:** Use Polars' native `read_database_uri()` to extract the data at blazing speed (via ConnectorX or ADBC), then pass the dataframe immediately into `proc.run()`. No scaffolding or temporary files required.

In [13]:
# ── Define the schema boundary for Postgres ─────────────
postgres_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: pg_users
info:
  title: bronze_pg_users

source:
  type: database
  path: sqlite:///{abs_db_path}

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    - name: signup_date
      type: string

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%'"
""",
    "06_integrations_demo/postgres_users.yaml",
)

print("Contract written for Postgres extraction")

Contract written for Postgres extraction


In [14]:
# ── Extract directly from DataProcessor ────
# By calling run_source() with engine=ENGINE, LakeLogic routes automatically to pl.read_database_uri.
proc = ll.DataProcessor("06_integrations_demo/postgres_users.yaml", engine=ENGINE)
res = proc.run_source()
res.good, res.bad = s.to_polars(res.good), s.to_polars(res.bad)

print(f"\nExtracted {res.source_count} rows from the db.")
print(f"Good rows: {res.good_count}")
print(f"Quarantined rows: {res.bad_count}")
display(res.good)

2026-04-28 06:53:07.281 | INFO     | lakelogic.core.processor:_run_database_source:4195 - Running database source for contract: bronze_pg_users via engine=polars
2026-04-28 06:53:07.281 | INFO     | lakelogic.core.processor:_run_database_source:4225 - Column projection: selecting 3 fields from contract model
2026-04-28 06:53:14.931 | INFO     | lakelogic.core.processor:_run_database_source:4350 - database: loaded 2 rows, 3 columns
2026-04-28 06:53:14.937 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 2 | Total: 2 | Good: 1 | Quarantine: 1 | Ratio: 50.00%



Extracted 2 rows from the db.
Good rows: 1
Quarantined rows: 1


id,email,signup_date
i64,str,str
1,"""test@example.com""","""2024-01-01"""


---
## 5. Incremental CDC (Change Data Capture)

**The Problem:** Your database has millions of rows. Extracting the full table every minute crushes the database and your pipeline.

**The Solution:** Set `load_mode: incremental` and a `watermark_field`. LakeLogic stores the max watermark safely in the `.lakelogic` state folder and dynamically injects `WHERE updated_at > last_watermark` into the SQL engine before data is even loaded.

In [15]:
# ── Contract declaring Incremental CDC ─────────────
cdc_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: cdc_orders
info:
  title: bronze_cdc_orders
  target_layer: bronze

source:
  type: database
  load_mode: incremental
  watermark_field: updated_at
  path: sqlite:///{abs_db_path}

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: updated_at
      type: timestamp
      required: true
""",
    "06_integrations_demo/cdc_orders.yaml",
)

print("Contract written with Incremental CDC watermark tracking!")

Contract written with Incremental CDC watermark tracking!


In [16]:
# ── Execute Incremental CDC Load ────
proc = ll.DataProcessor("06_integrations_demo/cdc_orders.yaml", engine=ENGINE)
res = proc.run_source()
res.good, res.bad = s.to_polars(res.good), s.to_polars(res.bad)

print(f"\nTotal processed: {res.source_count}")
display(res.good.head(3))

2026-04-28 06:53:14.962 | INFO     | lakelogic.core.processor:_run_database_source:4195 - Running database source for contract: bronze_cdc_orders via engine=polars
2026-04-28 06:53:14.964 | INFO     | lakelogic.core.processor:_run_database_source:4225 - Column projection: selecting 2 fields from contract model
2026-04-28 06:53:14.965 | INFO     | lakelogic.core.processor:_run_database_source:4252 - Incremental mode: First run detected. Running full table extraction.
2026-04-28 06:53:14.968 | INFO     | lakelogic.core.processor:_run_database_source:4350 - database: loaded 2 rows, 2 columns
2026-04-28 06:53:14.972 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=bronze] | Source: 2 | Total: 2 | Good: 2 | Quarantine: 0 | Ratio: 0.00%



Total processed: 2


id,updated_at
i64,datetime[μs]
1,2024-04-10 12:00:00
2,2024-04-12 12:00:00


---
## 6. Massive Initial Loads / Batch Ingestion

**The Problem:** You have a 100GB table in SQL Server or MySQL. Doing a `SELECT *` for the initial load will crash the worker node with an Out-of-Memory (OOM) error before it can evaluate any quality rules.

**The Solution:** Add `options: {fetch_size: N}`. LakeLogic automatically alters the execution path to use a streaming SQL iterator (via SQLAlchemy). It pulls 500,000 rows at a time, validates them against your contract, drops quarantined rows, writes the valid rows dynamically to your `target_layer`, and continues. Memory usage stays flat.

In [17]:
# ── Native Engine Dialect Agnosticism & Batching ─────
# LakeLogic automatically detects dialect extensions via URI (MySQL, SQL Server, SQLite).
batch_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: massive_orders
info:
  title: bronze_massive_orders

source:
  type: database
  path: sqlite:///{abs_db_path}
  options:
    fetch_size: 500000   # <- Subverts memory limits by iterating chunks natively

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: total
      type: float
""",
    "06_integrations_demo/batch_orders.yaml",
)

print("Contract written with fetch_size configured for batch streams!")

Contract written with fetch_size configured for batch streams!


In [32]:
# ── Execute Batch Ingestion (Iterative) ────
proc = ll.DataProcessor("06_integrations_demo/batch_orders.yaml", engine=ENGINE)
res = proc.run_source()
res.good, res.bad = s.to_polars(res.good), s.to_polars(res.bad)

display(res.good.head(3))

2026-04-28 06:55:54.760 | INFO     | lakelogic.core.processor:_run_database_source:4195 - Running database source for contract: bronze_massive_orders via engine=polars
2026-04-28 06:55:54.760 | INFO     | lakelogic.core.processor:_run_database_source:4225 - Column projection: selecting 2 fields from contract model
2026-04-28 06:55:54.761 | INFO     | lakelogic.core.processor:_run_database_source:4257 - Batch execution active. Fetching data in 500000 row chunks via SQLAlchemy yield.
2026-04-28 06:55:54.766 | INFO     | lakelogic.core.processor:_run_database_source:4273 - Processing database chunk 1 (1000 rows)...
2026-04-28 06:55:54.774 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 1000 | Total: 1000 | Good: 1000 | Quarantine: 0 | Ratio: 0.00%
2026-04-28 06:55:54.776 | INFO     | lakelogic.core.processor:_run_database_source:4285 - Batch ingestion complete across 1 chunks. Good: 1000, Bad: 0


id,total
i64,f64
1,1.5
2,3.0
3,4.5


---
## 7. Smart Column Projection (Pushdown)

**The Problem:** The source `orders` table has 150 columns (including heavy JSON blobs), but your analytics pipeline only needs 3 columns. Doing `SELECT *` wastes network bandwidth and memory.

**The Solution:** You do absolutely nothing! LakeLogic's Native Database engines intelligently read your `model.fields` list and dynamically construct precise `SELECT "col1", "col2"` queries. It only extracts exactly what is defined in the contract.

In [19]:
# ── Automatic Projection Pushdown ──────
# This contract will automatically generate the query:
# SELECT "order_id", "total_amount" FROM "wide_orders_table"
projection_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: wide_orders_table
info:
  title: bronze_narrow_orders

source:
  type: database
  path: sqlite:///{abs_db_path}

model:
  fields:
    # Only these two fields are extracted over the wire!
    - name: order_id
      type: string
    - name: total_amount
      type: float
""",
    "06_integrations_demo/projection_orders.yaml",
)

print("Contract written showcasing smart zero-effort projection pushdown!")

Contract written showcasing smart zero-effort projection pushdown!


In [20]:
# ── Execute Smart Projection Extraction ────
proc = ll.DataProcessor("06_integrations_demo/projection_orders.yaml", engine=ENGINE)
res = proc.run_source()
res.good, res.bad = s.to_polars(res.good), s.to_polars(res.bad)

print(f"\nTotal processed: {res.source_count}")
display(res.good.head(3))

2026-04-28 06:53:16.856 | INFO     | lakelogic.core.processor:_run_database_source:4195 - Running database source for contract: bronze_narrow_orders via engine=polars
2026-04-28 06:53:16.857 | INFO     | lakelogic.core.processor:_run_database_source:4225 - Column projection: selecting 2 fields from contract model
2026-04-28 06:53:16.859 | INFO     | lakelogic.core.processor:_run_database_source:4350 - database: loaded 2 rows, 2 columns
2026-04-28 06:53:16.862 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 2 | Total: 2 | Good: 2 | Quarantine: 0 | Ratio: 0.00%



Total processed: 2


order_id,total_amount
str,f64
"""A1""",100.5
"""A2""",50.0


---
## 8. Pre-Phase Column Filtering (Files & APIs)

**The Problem:** Section 7 showed Smart Column Projection for **databases**, where the SQL query is pushed down to the database server. But what about **files** (CSV, Parquet, JSON in S3/ADLS) and **REST APIs** (via dlt)? You can't send a `SELECT` statement to an S3 bucket.

**The Solution:** Use a `phase: pre` transformation. LakeLogic first downloads/reads the full file into an in-memory table, then runs your SQL **before** schema validation or quality rules fire. This gives you the same wildcard/prefix filtering power across *any* source type.

In [21]:
# ── Pre-Phase Column Filtering ────────────────────────────────
# For files and APIs, use pre-phase SQL to achieve the same
# wildcard/prefix column selection that databases get natively.
#
# Use case: Ingest a wide CSV/Parquet file but only keep columns
# matching a prefix (e.g. 'ice_') or exclude columns with a
# keyword (e.g. '_ex').

pre_filter_contract_yaml = """
version: 1.0.0
dataset: filtered_orders
info:
  title: orders_prefix_filtered

source:
  type: landing
  path: 06_integrations_demo/wide_orders.csv

# This runs AFTER the file is loaded into memory,
# but BEFORE schema validation or quality rules fire.
transformations:
  - phase: pre
    sql: >
      SELECT order_id, status,
             COLUMNS('ice_.*')
      FROM source
      
# For API sources (dlt), the exact same pattern works:
#   source:
#     type: dlt
#     dlt:
#       source: stripe_analytics
#       resource: charges
#   transformations:
#     - phase: pre
#       sql: "SELECT id, amount, COLUMNS('ice_.*') FROM source"

model:
  fields:
    - name: order_id
      type: string
      required: true
    - name: status
      type: string

materialization:
  format: parquet
  target_path: 06_integrations_demo/output/filtered_orders
"""

# Write the contract
s.write_contract(pre_filter_contract_yaml, "06_integrations_demo/pre_filter_orders.yaml")
print("Contract written: 06_integrations_demo/pre_filter_orders.yaml")

# Generate a wide CSV with 'ice_' prefixed columns to demonstrate
import csv
import os

os.makedirs("06_integrations_demo", exist_ok=True)
with open("06_integrations_demo/wide_orders.csv", "w", newline="") as f:
    writer = csv.writer(f)
    # 10 columns: 2 normal + 4 'ice_' prefix + 4 '_ex' suffix
    headers = [
        "order_id",
        "status",
        "ice_region",
        "ice_category",
        "ice_score",
        "ice_flag",
        "amount_ex",
        "tax_ex",
        "notes_ex",
        "internal_ex",
    ]
    writer.writerow(headers)
    writer.writerow(["ORD-001", "shipped", "EU", "electronics", "92", "true", "100", "20", "n/a", "debug"])
    writer.writerow(["ORD-002", "pending", "US", "apparel", "87", "false", "50", "10", "n/a", "test"])
    writer.writerow(["ORD-003", "delivered", "APAC", "food", "95", "true", "200", "40", "n/a", "prod"])

print("\nGenerated wide_orders.csv with 10 columns:")
print("  Normal:  order_id, status")
print("  ice_*:   ice_region, ice_category, ice_score, ice_flag")
print("  *_ex:    amount_ex, tax_ex, notes_ex, internal_ex")
print("\nThe pre-phase SQL will SELECT only order_id, status, and ice_* columns.")
print("The *_ex columns will be automatically excluded before validation.")

Contract written: 06_integrations_demo/pre_filter_orders.yaml

Generated wide_orders.csv with 10 columns:
  Normal:  order_id, status
  ice_*:   ice_region, ice_category, ice_score, ice_flag
  *_ex:    amount_ex, tax_ex, notes_ex, internal_ex

The pre-phase SQL will SELECT only order_id, status, and ice_* columns.
The *_ex columns will be automatically excluded before validation.


### When to Use Which Approach

| Source Type | Method | Where SQL Runs |
| :--- | :--- | :--- |
| **Database** (PostgreSQL, Snowflake, etc.) | `source.query:` | On the **database server** (network-efficient) |
| **Files** (CSV, Parquet, JSON in S3/ADLS) | `transformations: [{phase: pre, sql: ...}]` | In **LakeLogic's engine** (DuckDB/Polars/Spark) |
| **APIs** (via dlt — Stripe, Shopify, etc.) | `transformations: [{phase: pre, sql: ...}]` | In **LakeLogic's engine** (DuckDB/Polars/Spark) |

> **Key Insight:** For databases, push filtering to the server to save bandwidth. For files and APIs, LakeLogic applies the same filtering in-memory after download but before validation.

---
## 8. Cloud Data Sources (Azure, AWS, GCP)

**The Problem:** Your data lives in cloud storage — Azure Data Lake (ADLS), Amazon S3, or Google Cloud Storage. You need to read, validate, and materialize it without writing boilerplate credential code.

**The Solution:** LakeLogic natively resolves `abfss://`, `s3://`, and `gs://` URIs. Its built-in `CloudCredentialResolver` automatically detects credentials from environment variables, service principals, IAM roles, or `az login` — zero manual `storage_options` configuration.

In [22]:
# ── LakeLogic's built-in cloud credential resolver ──────────
from lakelogic import CloudCredentialResolver

resolver = CloudCredentialResolver()

# Auto-detects from env vars, az login, managed identity, or IAM roles
print("CloudCredentialResolver supports:")
print("  • Azure ADLS/Blob  — abfss://container@account.dfs.core.windows.net/")
print("  • Amazon S3        — s3://bucket/prefix/")
print("  • Google Cloud GCS — gs://bucket/prefix/")
print()
print("Authentication priority (Azure):")
print("  1. Explicit token/key (AZURE_STORAGE_ACCOUNT_KEY, SAS_TOKEN)")
print("  2. Service Principal (AZURE_CLIENT_ID + SECRET + TENANT_ID)")
print("  3. Account key from env var")
print("  4. Azure AD (az login / managed identity / workload identity)")
print()
print("Authentication priority (AWS):")
print("  1. Explicit credentials (AWS_ACCESS_KEY_ID + SECRET)")
print("  2. Environment variables")
print("  3. IAM role (boto3 default credential chain)")
print()
print("Authentication priority (GCP):")
print("  1. GOOGLE_SERVICE_ACCOUNT")
print("  2. GOOGLE_APPLICATION_CREDENTIALS env var")
print("  3. Application Default Credentials (gcloud auth)")

CloudCredentialResolver supports:
  • Azure ADLS/Blob  — abfss://container@account.dfs.core.windows.net/
  • Amazon S3        — s3://bucket/prefix/
  • Google Cloud GCS — gs://bucket/prefix/

Authentication priority (Azure):
  1. Explicit token/key (AZURE_STORAGE_ACCOUNT_KEY, SAS_TOKEN)
  2. Service Principal (AZURE_CLIENT_ID + SECRET + TENANT_ID)
  3. Account key from env var
  4. Azure AD (az login / managed identity / workload identity)

Authentication priority (AWS):
  1. Explicit credentials (AWS_ACCESS_KEY_ID + SECRET)
  2. Environment variables
  3. IAM role (boto3 default credential chain)

Authentication priority (GCP):
  1. GOOGLE_SERVICE_ACCOUNT
  2. GOOGLE_APPLICATION_CREDENTIALS env var
  3. Application Default Credentials (gcloud auth)


In [23]:
# ── Azure Data Lake Storage (ADLS Gen2) ─────────────────
# Just set your source.path to an abfss:// URI and LakeLogic handles the rest.
# Credentials are auto-resolved from `az login` or env vars.
#
# The `partition` block is the key optimization:
#   - format: maps to your landing directory structure (strftime tokens)
#   - lookback_days: only scans the last N days instead of the entire lake
#   - This turns a full glob scan into a precise directory lookup

import os

# ── Production: pull from environment variables ─────────────────────
AZURE_STORAGE_ACCOUNT = os.environ.get("AZURE_STORAGE_ACCOUNT", "mystorageaccount")
AZURE_LANDING_CONTAINER = os.environ.get("AZURE_LANDING_CONTAINER", "landing")
AZURE_SILVER_CONTAINER = os.environ.get("AZURE_SILVER_CONTAINER", "silver")

azure_contract_yaml = f"""
version: 1.0.0
dataset: customer_events
info:
  title: bronze_customer_events
  domain: marketing
  target_layer: bronze

source:
  path: abfss://{AZURE_LANDING_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events
  format: parquet
  load_mode: incremental
  watermark_strategy: pipeline_log

  # Partition-aware ingestion — only scan relevant date directories
  partition:
    format: "y_%Y/m_%m/d_%d"        # matches: events/y_2026/m_04/d_16/*.parquet
    lookback_days: 3                  # only scan last 3 days (not the entire lake)
    # start_date: "2026-01-01"        # optional: override for backfills
    # end_date: "2026-01-31"          # optional: override for backfills

model:
  fields:
    - name: event_id
      type: string
      required: true
    - name: customer_id
      type: string
      required: true
    - name: event_type
      type: string
    - name: timestamp
      type: timestamp
      required: true

quality:
  row_rules:
    - name: valid_event
      sql: "event_type IN ('click', 'view', 'purchase', 'signup')"

materialization:
  strategy: append
  format: delta
  partition_by: [event_type]           # Delta table partitioned for fast downstream queries
  target_path: abfss://{AZURE_SILVER_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events

server:
  type: local
  path: "."
"""

print("Azure ADLS contract with partition-aware ingestion:")
print(f"  Storage account: {AZURE_STORAGE_ACCOUNT}")
print(f"  Landing:         abfss://{AZURE_LANDING_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events")
print(f"  Output:          abfss://{AZURE_SILVER_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events")
print()
print("  Landing structure:")
print(f"    abfss://{AZURE_LANDING_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events/")
print("      y_2026/")
print("        m_04/")
print("          d_14/*.parquet")
print("          d_15/*.parquet")
print("          d_16/*.parquet    <-- only these 3 days scanned (lookback_days: 3)")
print()
print("  Without partition:  scans ALL directories (slow, expensive)")
print("  With partition:     scans only y_2026/m_04/d_14, d_15, d_16 (fast, cheap)")
print()
print("# To execute:")
print('# proc = ll.DataProcessor(contract, engine=ENGINE)')
print("# good, bad = proc.run_source()  # Reads only last 3 days from ADLS")

Azure ADLS contract with partition-aware ingestion:
  Storage account: mystorageaccount
  Landing:         abfss://landing@mystorageaccount.dfs.core.windows.net/events
  Output:          abfss://silver@mystorageaccount.dfs.core.windows.net/events

  Landing structure:
    abfss://landing@mystorageaccount.dfs.core.windows.net/events/
      y_2026/
        m_04/
          d_14/*.parquet
          d_15/*.parquet
          d_16/*.parquet    <-- only these 3 days scanned (lookback_days: 3)

  Without partition:  scans ALL directories (slow, expensive)
  With partition:     scans only y_2026/m_04/d_14, d_15, d_16 (fast, cheap)

# To execute:
# proc = ll.DataProcessor(contract, engine="polars")
# good, bad = proc.run_source()  # Reads only last 3 days from ADLS


In [24]:
# ── Amazon S3 ───────────────────────────────────────
# Same pattern — just swap the URI to s3://

import os

# ── Production: pull from environment variables ─────────────────────
AWS_S3_BUCKET = os.environ.get("AWS_S3_BUCKET", "my-data-lake")
AWS_S3_PREFIX = os.environ.get("AWS_S3_PREFIX", "landing/orders")

s3_contract_yaml = f"""
version: 1.0.0
dataset: order_events
info:
  title: bronze_order_events
  domain: commerce
  target_layer: bronze

source:
  path: s3://{AWS_S3_BUCKET}/{AWS_S3_PREFIX}/*.json
  format: json
  load_mode: incremental
  watermark_strategy: source_mtime

model:
  fields:
    - name: order_id
      type: string
      required: true
    - name: customer_id
      type: string
      required: true
    - name: total_amount
      type: float
    - name: currency
      type: string
    - name: created_at
      type: timestamp
      required: true

quality:
  row_rules:
    - name: positive_amount
      sql: "total_amount > 0"
    - name: valid_currency
      sql: "currency IN ('USD', 'EUR', 'GBP', 'JPY')"

server:
  type: local
  path: "."
"""

print("AWS S3 contract (would run with real credentials):")
print(f"  source: s3://{AWS_S3_BUCKET}/{AWS_S3_PREFIX}/*.json")
print()
print("# To execute:")
print('# contract = s.write_contract(s3_contract_yaml, "commerce/orders.yaml")')
print('# proc = ll.DataProcessor(contract, engine=ENGINE)')
print("# good, bad = proc.run_source()  # Reads from S3 automatically")

AWS S3 contract (would run with real credentials):
  source: s3://my-data-lake/landing/orders/*.json

# To execute:
# contract = s.write_contract(s3_contract_yaml, "commerce/orders.yaml")
# proc = ll.DataProcessor(contract, engine="polars")
# good, bad = proc.run_source()  # Reads from S3 automatically


In [25]:
# ── Google Cloud Storage (GCS) ─────────────────────
# Same pattern — just swap to gs://

import os

# ── Production: pull from environment variables ─────────────────────
GCS_BUCKET = os.environ.get("GCS_BUCKET", "my-analytics-bucket")
GCS_PREFIX = os.environ.get("GCS_PREFIX", "sessions")

gcs_contract_yaml = f"""
version: 1.0.0
dataset: user_sessions
info:
  title: bronze_user_sessions
  domain: analytics
  target_layer: bronze

source:
  path: gs://{GCS_BUCKET}/{GCS_PREFIX}/*.parquet
  format: parquet
  load_mode: full

model:
  fields:
    - name: session_id
      type: string
      required: true
    - name: user_id
      type: string
      required: true
    - name: page_views
      type: integer
    - name: duration_seconds
      type: integer
    - name: started_at
      type: timestamp

quality:
  row_rules:
    - name: positive_views
      sql: "page_views >= 0"
    - name: reasonable_duration
      sql: "duration_seconds BETWEEN 0 AND 86400"

server:
  type: local
  path: "."
"""

print("GCP Cloud Storage contract (would run with real credentials):")
print(f"  source: gs://{GCS_BUCKET}/{GCS_PREFIX}/*.parquet")
print()
print("# To execute:")
print('# contract = s.write_contract(gcs_contract_yaml, "analytics/sessions.yaml")')
print('# proc = ll.DataProcessor(contract, engine=ENGINE)')
print("# good, bad = proc.run_source()  # Reads from GCS automatically")

GCP Cloud Storage contract (would run with real credentials):
  source: gs://my-analytics-bucket/sessions/*.parquet

# To execute:
# contract = s.write_contract(gcs_contract_yaml, "analytics/sessions.yaml")
# proc = ll.DataProcessor(contract, engine="polars")
# good, bad = proc.run_source()  # Reads from GCS automatically


In [26]:
# ── Databricks Secret Scope Integration ─────────────────
# On Databricks, LakeLogic can pull credentials from secret scopes
# (backed by Azure Key Vault, AWS Secrets Manager, etc.)
from lakelogic.engines.cloud_credentials import DatabricksSecretResolver

print("DatabricksSecretResolver usage (Databricks notebooks only):")
print()
print("# Azure (Key Vault-backed scope):")
print('# resolver = DatabricksSecretResolver.for_cloud("azure", scope="lakelogic")')
print("# options  = resolver.resolve_storage_options(")
print('#     "abfss://silver@myaccount.dfs.core.windows.net/orders/"')
print("# )")
print()
print("# AWS (Secrets Manager-backed scope):")
print('# resolver = DatabricksSecretResolver.for_cloud("aws", scope="lakelogic-aws")')
print('# options  = resolver.resolve_storage_options("s3://my-bucket/silver/")')
print()
print("# GCP (Secret Manager-backed scope):")
print('# resolver = DatabricksSecretResolver.for_cloud("gcp", scope="lakelogic-gcp")')
print('# options  = resolver.resolve_storage_options("gs://my-bucket/silver/")')

DatabricksSecretResolver usage (Databricks notebooks only):

# Azure (Key Vault-backed scope):
# resolver = DatabricksSecretResolver.for_cloud("azure", scope="lakelogic")
# options  = resolver.resolve_storage_options(
#     "abfss://silver@myaccount.dfs.core.windows.net/orders/"
# )

# AWS (Secrets Manager-backed scope):
# resolver = DatabricksSecretResolver.for_cloud("aws", scope="lakelogic-aws")
# options  = resolver.resolve_storage_options("s3://my-bucket/silver/")

# GCP (Secret Manager-backed scope):
# resolver = DatabricksSecretResolver.for_cloud("gcp", scope="lakelogic-gcp")
# options  = resolver.resolve_storage_options("gs://my-bucket/silver/")


## What You Just Saw

- **dbt adapter** — import `schema.yml` as a LakeLogic contract, zero duplication
- **dlt adapter** — declare the API in the contract, `run_source()` does extraction + validation
- **Native streaming connectors** — `WebSocketConnector` fetches live BTC trades, pre-validation `rename` transformations map cryptic field names to business-friendly columns
- **Native database (Polars)** — `pl.read_database_uri()` for high-speed SQL extraction with contract validation
- **Incremental CDC** — watermark-based change tracking with automatic state management
- **Batch ingestion** — `fetch_size` for memory-safe chunked processing of massive tables
- **Smart projection pushdown** — automatic `SELECT "col"` extraction based purely on the contract schema
- **Pre-phase column filtering** — wildcard/prefix column selection for files and APIs using `phase: pre` SQL transformations
- **Cloud data sources** — native `abfss://`, `s3://`, `gs://` URI support with automatic credential resolution (Azure AD, IAM roles, service principals, Key Vault)
- **Same reconciliation guarantee** — every row accounted for regardless of source

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.